<a href="https://colab.research.google.com/github/gunnsmart/upscaly-colab-version-/blob/arena%2F01a0fb33-upscaly-colab-version/upscayl_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Upscayl Studio บน Google Colab

ใช้งาน **Upscayl NCNN/Vulkan backend ทางการ** ผ่านหน้า UI ภายใน Notebook — เลือกไฟล์, โมเดล, scale, format และ tile แล้วกด Upscale ได้เลย พร้อมดูภาพก่อน/หลังและดาวน์โหลดผลทั้งหมดเป็น ZIP

> ต้องเลือก Colab runtime ที่มี **NVIDIA GPU** ก่อนรัน เซลล์ติดตั้งจะเตรียม/ตรวจ Vulkan และหยุดพร้อมคำแนะนำหาก runtime ไม่มี NVIDIA Vulkan device

## เริ่มใช้งาน

1. เลือก **Runtime → Change runtime type → GPU**
2. ตั้งค่า Google Drive ได้ในเซลล์ **ตั้งค่า workspace** (ไม่บังคับ)
3. รันทุกเซลล์ตามลำดับ; หน้า UI จะปรากฏก่อน จากนั้น Notebook จะเตรียม backend อยู่ด้านล่าง
4. เลือกหรือวางรูป, ตั้งค่า แล้วกด **เริ่ม Upscale**
5. ดูตัวอย่างผลและกด **ดาวน์โหลด ZIP** — ไฟล์เต็มจะถูกบันทึกในโฟลเดอร์ `output`

รูปประมวลผลภายใน Colab runtime ของคุณ ไม่ต้องสร้าง public share link


In [ ]:
from pathlib import Path

# ตั้งเป็น True เพื่อเก็บ input/output ไว้ใน Google Drive
USE_GOOGLE_DRIVE = False
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORKSPACE = Path("/content/drive/MyDrive/Upscayl-Colab")
else:
    WORKSPACE = Path("/content/upscayl-colab")

INPUT_DIR = WORKSPACE / "input"
OUTPUT_DIR = WORKSPACE / "output"
RUNTIME_DIR = Path("/content/upscayl-colab-runtime")
BIN_DIR = RUNTIME_DIR / "bin"
BIN_PATH = BIN_DIR / "upscayl-bin"
MODEL_DIR = RUNTIME_DIR / "models"  # Upscayl backend ต้องการโฟลเดอร์ชื่อ models
for folder in (INPUT_DIR, OUTPUT_DIR, BIN_DIR, MODEL_DIR):
    folder.mkdir(parents=True, exist_ok=True)

print(f"Workspace: {WORKSPACE}")
print(f"Output จะถูกเก็บที่: {OUTPUT_DIR}")


## หน้าจอ Upscayl Studio

แผง UI จะแสดงทันทีหลังรัน cell นี้ ขณะเดียวกันเซลล์ backend ด้านล่างจะติดตั้ง dependencies และตรวจ Vulkan; หาก runtime ใช้งาน Vulkan ไม่ได้ UI ยังอยู่ให้เห็น แต่ปุ่มประมวลผลจะแจ้งสาเหตุแทน


In [ ]:
# Interactive Upscayl panel for Google Colab
import base64
import datetime
import html
import io
import os
import subprocess
import sys
import traceback
import zipfile
from pathlib import Path

try:
    import ipywidgets as widgets
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ipywidgets"])
    import ipywidgets as widgets

from google.colab import files, output as colab_output
from IPython.display import HTML, clear_output, display
from PIL import Image

# Colab's widget manager enables the native upload/control widgets in the notebook.
try:
    colab_output.enable_custom_widget_manager()
except Exception:
    pass

MODEL_LABELS = {
    "upscayl-standard-4x": "Standard — ภาพทั่วไป (แนะนำ)",
    "upscayl-lite-4x": "Lite — โมเดลเล็ก/เร็ว",
    "high-fidelity-4x": "High Fidelity — เน้นรายละเอียด",
    "remacri-4x": "Remacri — รายละเอียดและ texture",
    "ultramix-balanced-4x": "Ultramix Balanced — สมดุล",
    "ultrasharp-4x": "Ultrasharp — เน้นความคม",
    "digital-art-4x": "Digital Art — ภาพวาด/กราฟิก",
}
MODEL_REF = "a00d55fee90e0f9435d5eaa86e76700df8199af8"
MODEL_BASE_URL = (
    "https://raw.githubusercontent.com/upscayl/upscayl/"
    f"{MODEL_REF}/resources/models"
)
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp"}

model_dropdown = widgets.Dropdown(
    options=[(label, model_id) for model_id, label in MODEL_LABELS.items()],
    value="upscayl-standard-4x",
    layout=widgets.Layout(width="100%"),
)
scale_dropdown = widgets.Dropdown(
    options=[("2×", 2), ("3×", 3), ("4×", 4)],
    value=4,
    layout=widgets.Layout(width="100%"),
)
format_dropdown = widgets.Dropdown(
    options=[("PNG — แนะนำ/รองรับโปร่งใส", "png"), ("JPG", "jpg"), ("WebP", "webp")],
    value="png",
    layout=widgets.Layout(width="100%"),
)
tile_dropdown = widgets.Dropdown(
    options=[("Auto", 0), ("32 — VRAM ต่ำ", 32), ("64", 64), ("128", 128)],
    value=0,
    layout=widgets.Layout(width="100%"),
)
tta_checkbox = widgets.Checkbox(
    value=False,
    description="เปิด TTA (ช้าลง)",
    indent=False,
    layout=widgets.Layout(width="auto"),
)
drive_checkbox = widgets.Checkbox(
    value=False,
    description="รวมรูปที่อยู่ในโฟลเดอร์ input",
    indent=False,
    layout=widgets.Layout(width="auto"),
)
upload_widget = widgets.FileUpload(
    accept=".png,.jpg,.jpeg,.webp",
    multiple=True,
    description="เลือกภาพ",
    button_style="primary",
    icon="upload",
    layout=widgets.Layout(width="100%"),
)
run_button = widgets.Button(
    description="เริ่ม Upscale",
    button_style="primary",
    icon="bolt",
    layout=widgets.Layout(width="190px", height="42px"),
)
download_button = widgets.Button(
    description="ดาวน์โหลด ZIP",
    button_style="success",
    icon="download",
    disabled=True,
    layout=widgets.Layout(width="190px", height="42px"),
)
progress = widgets.IntProgress(
    value=0,
    min=0,
    max=1,
    description="พร้อม",
    bar_style="info",
    layout=widgets.Layout(width="100%"),
)
status = widgets.HTML()
gallery = widgets.Output(
    layout=widgets.Layout(width="100%", border="1px solid #e5e7eb", padding="10px")
)
logs = widgets.Output(
    layout=widgets.Layout(
        width="100%", max_height="240px", overflow_y="auto",
        border="1px solid #e5e7eb", padding="8px"
    )
)
state = {"results": [], "archive": None}


def set_status(message, kind="info"):
    palette = {
        "info": ("#eff6ff", "#1d4ed8"),
        "working": ("#fff7ed", "#c2410c"),
        "success": ("#ecfdf5", "#047857"),
        "error": ("#fef2f2", "#b91c1c"),
    }
    background, color = palette.get(kind, palette["info"])
    status.value = (
        f'<div style="background:{background};color:{color};padding:10px 12px;'
        f'border-radius:10px;font-weight:600">{html.escape(str(message))}</div>'
    )


def read_uploaded_files():
    """Handle ipywidgets 7's dict and ipywidgets 8's tuple upload formats."""
    value = upload_widget.value
    records = []

    def add_record(name, record):
        if isinstance(record, dict):
            content = record.get("content")
            if content is None:
                content = record.get("data")
            metadata = record.get("metadata") or {}
            name = name or record.get("name") or metadata.get("name")
        else:
            content = record
        if not name or content is None:
            return
        if isinstance(content, memoryview):
            content = content.tobytes()
        else:
            content = bytes(content)
        clean_name = Path(str(name).replace("\\", "/")).name
        if Path(clean_name).suffix.lower() not in IMAGE_EXTENSIONS:
            return
        records.append((clean_name, content))

    if isinstance(value, dict):
        for name, record in value.items():
            add_record(name, record)
    elif isinstance(value, (tuple, list)):
        for record in value:
            add_record(None, record)
    return records


def ensure_model(model_id):
    for suffix, min_bytes in ((".param", 1_000), (".bin", 100_000)):
        target = MODEL_DIR / f"{model_id}{suffix}"
        if target.is_file() and target.stat().st_size >= min_bytes:
            continue
        if target.exists():
            target.unlink()
        url = f"{MODEL_BASE_URL}/{model_id}{suffix}"
        with logs:
            print(f"กำลังดาวน์โหลดโมเดล {target.name} ...")
        download_file(url, target)
        if not target.is_file() or target.stat().st_size < min_bytes:
            raise RuntimeError(f"ไฟล์โมเดล {target.name} ดาวน์โหลดไม่สมบูรณ์")


def thumbnail_data_uri(path, max_side=480):
    with Image.open(path) as source_image:
        image = source_image.convert("RGBA")
        background = Image.new("RGBA", image.size, (255, 255, 255, 255))
        background.alpha_composite(image)
        image = background.convert("RGB")
        image.thumbnail((max_side, max_side))
        buffer = io.BytesIO()
        image.save(buffer, format="JPEG", quality=82, optimize=True)
    encoded = base64.b64encode(buffer.getvalue()).decode("ascii")
    return f"data:image/jpeg;base64,{encoded}"


def render_results(results):
    with gallery:
        clear_output(wait=True)
        if not results:
            display(HTML("<p>ยังไม่มีผลลัพธ์ให้แสดง</p>"))
            return
        cards = []
        for original, result_path, source_kind in results[:8]:
            original_uri = thumbnail_data_uri(original)
            result_uri = thumbnail_data_uri(result_path)
            original_name = html.escape(original.name)
            result_name = html.escape(result_path.name)
            source_label = "Drive/input" if source_kind == "drive" else "อัปโหลด"
            cards.append(
                '<article style="background:#fff;border:1px solid #e5e7eb;border-radius:14px;'
                'padding:12px;box-shadow:0 2px 8px rgba(15,23,42,.05);min-width:0">'
                f'<div style="font-weight:700;margin-bottom:4px">{original_name}</div>'
                f'<div style="color:#64748b;font-size:12px;margin-bottom:10px">{source_label}</div>'
                '<div style="display:grid;grid-template-columns:1fr 1fr;gap:8px">'
                f'<figure style="margin:0"><img src="{original_uri}" style="width:100%;height:auto;'
                'border-radius:8px;object-fit:contain"><figcaption style="font-size:12px;color:#64748b">ก่อน</figcaption></figure>'
                f'<figure style="margin:0"><img src="{result_uri}" style="width:100%;height:auto;'
                f'border-radius:8px;object-fit:contain"><figcaption style="font-size:12px;color:#64748b">หลัง · {result_name}</figcaption></figure>'
                '</div></article>'
            )
        extra = ""
        if len(results) > 8:
            extra = f'<p style="color:#64748b">แสดงตัวอย่าง 8 จาก {len(results)} ภาพ — ดาวน์โหลด ZIP เพื่อรับไฟล์ทั้งหมด</p>'
        display(HTML(
            '<div style="display:grid;grid-template-columns:repeat(auto-fit,minmax(300px,1fr));gap:12px">'
            + "".join(cards) + "</div>" + extra
        ))


def start_upscaling(_):
    run_button.disabled = True
    download_button.disabled = True
    state["results"] = []
    state["archive"] = None
    with gallery:
        clear_output(wait=True)
    with logs:
        clear_output(wait=True)
    progress.value = 0
    progress.max = 1
    progress.bar_style = "info"
    set_status("กำลังเตรียมงาน...", "working")

    try:
        if not globals().get("BACKEND_READY", False):
            detail = globals().get("BACKEND_ERROR") or "ยังไม่ได้รันเซลล์เตรียม backend"
            raise RuntimeError(f"Upscayl backend ยังไม่พร้อม: {detail}")
        uploads = read_uploaded_files()
        drive_images = []
        if drive_checkbox.value:
            drive_images = sorted(
                path for path in INPUT_DIR.rglob("*")
                if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
                and not path.name.startswith(".")
            )
        if not uploads and not drive_images:
            raise ValueError("กรุณาเลือกภาพ หรือเลือกให้รวมรูปจากโฟลเดอร์ input")

        job_id = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
        upload_dir = RUNTIME_DIR / "ui-uploads" / job_id
        output_job_dir = OUTPUT_DIR / job_id
        upload_dir.mkdir(parents=True, exist_ok=True)
        output_job_dir.mkdir(parents=True, exist_ok=True)

        jobs = []
        used_names = set()
        for index, (name, content) in enumerate(uploads, start=1):
            base_name = name
            if base_name.lower() in used_names:
                base_name = f"{index}_{base_name}"
            used_names.add(base_name.lower())
            source_path = upload_dir / base_name
            source_path.write_bytes(content)
            jobs.append((source_path, Path(base_name), "upload"))
        for source_path in drive_images:
            jobs.append((source_path, source_path.relative_to(INPUT_DIR), "drive"))

        selected_model = model_dropdown.value
        selected_scale = scale_dropdown.value
        selected_format = format_dropdown.value
        selected_tile = tile_dropdown.value
        use_tta = tta_checkbox.value
        ensure_model(selected_model)

        progress.max = len(jobs)
        progress.value = 0
        progress.description = f"0/{len(jobs)}"
        successes = []
        failures = []
        with logs:
            print(f"Upscayl: {MODEL_LABELS[selected_model]} | {selected_scale}x | GPU {GPU_ID}")
            print(f"กำลังประมวลผลทั้งหมด {len(jobs)} ภาพ\n")

        for index, (source_path, relative_path, source_kind) in enumerate(jobs, start=1):
            output_folder = output_job_dir / source_kind / relative_path.parent
            output_folder.mkdir(parents=True, exist_ok=True)
            destination = output_folder / (
                f"{relative_path.stem}_upscayl_{selected_scale}x_{selected_model}.{selected_format}"
            )
            command = [
                str(BIN_PATH),
                "-i", str(source_path),
                "-o", str(destination),
                "-s", str(selected_scale),
                "-m", str(MODEL_DIR),
                "-n", selected_model,
                "-g", str(GPU_ID),
                "-t", str(selected_tile),
                "-f", selected_format,
                "-v",
            ]
            if use_tta:
                command.append("-x")
            set_status(f"กำลัง Upscale {index}/{len(jobs)}: {relative_path.name}", "working")
            result = subprocess.run(
                command,
                text=True,
                capture_output=True,
                env=os.environ.copy(),
            )
            log = (result.stdout or "") + (result.stderr or "")
            if result.returncode != 0 or not destination.is_file() or destination.stat().st_size == 0:
                failures.append((source_path.name, log[-2500:] or f"CLI exit code {result.returncode}"))
                with logs:
                    print(f"❌ {relative_path.name}\n{log[-1200:]}\n")
            else:
                successes.append((source_path, destination, source_kind))
                with logs:
                    print(f"✅ {relative_path.name} → {destination.name}")
            progress.value = index
            progress.description = f"{index}/{len(jobs)}"

        state["results"] = successes
        if successes:
            archive_path = RUNTIME_DIR / f"upscayl-results-{job_id}.zip"
            with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_STORED) as archive:
                for _, result_path, _ in successes:
                    archive.write(result_path, arcname=str(result_path.relative_to(output_job_dir)))
            state["archive"] = archive_path
            download_button.disabled = False
            render_results(successes)
            message = f"เสร็จแล้ว {len(successes)}/{len(jobs)} ภาพ"
            if failures:
                message += f" (ล้มเหลว {len(failures)} ภาพ — ดูรายละเอียดใน log)"
            set_status(message, "success" if not failures else "info")
            progress.bar_style = "success" if not failures else "warning"
        else:
            set_status(f"ไม่สำเร็จทุกภาพ ({len(failures)} ภาพ) — ดูรายละเอียดใน log", "error")
            progress.bar_style = "danger"
    except Exception as error:
        set_status(str(error), "error")
        with logs:
            traceback.print_exc()
    finally:
        run_button.disabled = False


def download_results(_):
    archive_path = state.get("archive")
    if not archive_path or not Path(archive_path).is_file():
        set_status("ยังไม่มี ZIP ให้ดาวน์โหลด", "error")
        return
    files.download(str(archive_path))


run_button.on_click(start_upscaling)
download_button.on_click(download_results)

header = widgets.HTML(
    '<div style="background:linear-gradient(120deg,#111827,#312e81 65%,#4f46e5);'
    'color:white;padding:22px 24px;border-radius:16px;margin-bottom:4px">'
    '<div style="font-size:12px;letter-spacing:2px;opacity:.8">NCNN · VULKAN · GOOGLE COLAB</div>'
    '<h1 style="color:white;margin:6px 0;font-size:28px">Upscayl Studio</h1>'
    f'<div style="opacity:.88">GPU: {html.escape(str(globals().get("gpu_name", "ตรวจสอบสถานะ backend ด้านล่าง")))} · รูปอยู่ใน runtime ของคุณ</div>'
    '</div>'
)
settings_grid = widgets.GridBox(
    children=[
        widgets.VBox([widgets.HTML("<b>โมเดล AI</b>"), model_dropdown]),
        widgets.VBox([widgets.HTML("<b>ขนาดผลลัพธ์</b>"), scale_dropdown]),
        widgets.VBox([widgets.HTML("<b>รูปแบบไฟล์</b>"), format_dropdown]),
        widgets.VBox([widgets.HTML("<b>Tile size</b>"), tile_dropdown]),
    ],
    layout=widgets.Layout(
        width="100%",
        grid_template_columns="repeat(2, minmax(220px, 1fr))",
        grid_gap="12px",
    ),
)
if globals().get("BACKEND_READY", False):
    set_status("พร้อมใช้งาน — เลือกรูปและตั้งค่าก่อนกดเริ่ม", "info")
else:
    _backend_detail = globals().get("BACKEND_ERROR") or "ยังไม่ได้รันเซลล์เตรียม backend"
    set_status(f"UI พร้อมแล้ว แต่ backend ยังไม่พร้อม: {_backend_detail}", "error")
ui = widgets.VBox(
    [
        header,
        widgets.HTML("<h3 style='margin:14px 0 4px'>1 · เลือกรูป</h3><p style='color:#64748b;margin-top:0'>ลากรูปมาวางหรือกดเลือกได้หลายภาพ · รองรับ JPG, PNG, WebP</p>"),
        upload_widget,
        drive_checkbox,
        widgets.HTML("<h3 style='margin:14px 0 4px'>2 · ตั้งค่าภาพ</h3>"),
        settings_grid,
        tta_checkbox,
        widgets.HBox([run_button, download_button], layout=widgets.Layout(flex_flow="row wrap", grid_gap="10px", margin="10px 0")),
        progress,
        status,
        widgets.HTML("<h3 style='margin:18px 0 6px'>ตัวอย่างผลลัพธ์</h3>"),
        gallery,
        widgets.HTML("<h3 style='margin:18px 0 6px'>บันทึกการทำงาน</h3>"),
        logs,
    ],
    layout=widgets.Layout(width="100%", max_width="1100px", margin="0 auto", padding="8px"),
)
display(ui)


## เตรียม backend

เซลล์นี้ทำงานหลังหน้า UI แสดงแล้ว: ติดตั้ง dependency, ตรวจว่า Vulkan เห็น NVIDIA GPU และดาวน์โหลด CLI ทางการพร้อมตรวจ SHA-256 หาก setup ไม่ผ่านจะแสดงสาเหตุ แต่ UI จะยังคงอยู่


In [ ]:
BACKEND_READY = False
BACKEND_ERROR = None
try:
    # เตรียม NVIDIA Vulkan และดาวน์โหลด CLI ทางการของ Upscayl
    import glob
    import hashlib
    import json
    import os
    import re
    import shutil
    import stat
    import subprocess
    import urllib.request
    import zipfile
    from pathlib import Path


    def run_command(args, *, check=True, env=None):
        result = subprocess.run(args, text=True, capture_output=True, env=env)
        if check and result.returncode != 0:
            detail = (result.stdout or "") + (result.stderr or "")
            raise RuntimeError(
                f"คำสั่งล้มเหลว: {' '.join(map(str, args))}\n{detail[-5000:]}"
            )
        return result


    # อ่านเวอร์ชันไดรเวอร์ เพื่อติดตั้ง NVIDIA Vulkan ICD รุ่นเดียวกับ driver branch
    smi = run_command(
        ["nvidia-smi", "--query-gpu=driver_version,name", "--format=csv,noheader"],
        check=False,
    )
    if smi.returncode != 0 or not smi.stdout.strip():
        raise RuntimeError(
            "ไม่พบ NVIDIA GPU ใน runtime นี้ กรุณาเลือก Runtime → Change runtime type → GPU "
            "แล้วรันเซลล์ใหม่"
        )

    gpu_line = smi.stdout.strip().splitlines()[0]
    parts = [part.strip() for part in gpu_line.split(",", 1)]
    driver_version = parts[0]
    gpu_name = parts[1] if len(parts) > 1 else "NVIDIA GPU"
    major_match = re.match(r"(\d+)", driver_version)
    if not major_match:
        raise RuntimeError(f"อ่าน NVIDIA driver version ไม่ได้: {driver_version}")
    driver_major = major_match.group(1)
    print(f"NVIDIA GPU: {gpu_name} | driver {driver_version}")

    apt_update = run_command(["apt-get", "update", "-qq"], check=False)
    if apt_update.returncode != 0:
        raise RuntimeError(f"apt-get update ล้มเหลว:\n{apt_update.stderr[-3000:]}")


    def apt_package_available(package):
        policy = run_command(["apt-cache", "policy", package], check=False).stdout
        match = re.search(r"Candidate:\s*(\S+)", policy)
        return bool(match and match.group(1) != "(none)")


    packages = ["libvulkan1", "vulkan-tools"]
    matching_driver_package = f"libnvidia-gl-{driver_major}"
    if apt_package_available(matching_driver_package):
        packages.append(matching_driver_package)
    else:
        print(f"ไม่พบแพ็กเกจ {matching_driver_package} ใน apt; จะลองใช้ Vulkan ICD ที่มีอยู่ใน runtime")
    if apt_package_available("libomp5"):
        packages.append("libomp5")

    apt_install = run_command(["apt-get", "install", "-y", "-qq", *packages], check=False)
    if apt_install.returncode != 0:
        raise RuntimeError(
            f"ติดตั้ง Vulkan dependencies ไม่สำเร็จ ({', '.join(packages)}):\n"
            f"{apt_install.stdout[-2000:]}\n{apt_install.stderr[-3000:]}"
        )

    # บังคับใช้ NVIDIA ICD หากแพ็กเกจติดตั้งไฟล์ JSON ไว้แล้ว
    icd_candidates = []
    for icd_root in ("/usr/share/vulkan/icd.d", "/etc/vulkan/icd.d"):
        icd_candidates.extend(glob.glob(f"{icd_root}/*nvidia*.json"))
    NVIDIA_ICD = None
    for candidate in sorted(icd_candidates):
        try:
            with open(candidate, "r", encoding="utf-8") as handle:
                if "ICD" in json.load(handle):
                    NVIDIA_ICD = candidate
                    break
        except (OSError, json.JSONDecodeError):
            continue
    if NVIDIA_ICD:
        os.environ["VK_ICD_FILENAMES"] = NVIDIA_ICD
        os.environ["VK_DRIVER_FILES"] = NVIDIA_ICD
        print(f"NVIDIA Vulkan ICD: {NVIDIA_ICD}")
    else:
        print("ไม่พบไฟล์ NVIDIA ICD JSON; จะตรวจสอบ Vulkan device ที่ runtime เปิดให้ใช้")

    vulkan = run_command(["vulkaninfo", "--summary"], check=False, env=os.environ.copy())
    vulkan_report = (vulkan.stdout or "") + (vulkan.stderr or "")
    if vulkan.returncode != 0:
        raise RuntimeError(
            "Vulkan เริ่มทำงานไม่ได้ใน Colab runtime นี้\n"
            f"{vulkan_report[-5000:]}\n"
            "ตรวจว่าเลือก GPU runtime แล้ว restart runtime จากนั้นลองใหม่ "
            "(Colab บาง runtime อาจไม่มี Vulkan support)"
        )

    has_nvidia_device = bool(re.search(r"vendorID\s*=\s*0x10de\b", vulkan_report, re.IGNORECASE))
    if not has_nvidia_device:
        raise RuntimeError(
            "vulkaninfo ไม่พบ NVIDIA device (vendor ID 0x10de) จึงยังใช้ Upscayl NCNN/Vulkan ไม่ได้\n"
            f"{vulkan_report[:3000]}\n"
            "ลองเลือก GPU runtime ใหม่และ restart; การมี CUDA/nvidia-smi อย่างเดียวไม่ยืนยันว่า Vulkan ใช้งานได้"
        )
    print("✅ Vulkan ตรวจพบ NVIDIA GPU")


    def download_file(url, destination, expected_sha256=None):
        destination = Path(destination)
        destination.parent.mkdir(parents=True, exist_ok=True)
        if destination.is_file() and destination.stat().st_size > 0:
            if expected_sha256 is None:
                return destination
            current_hash = hashlib.sha256(destination.read_bytes()).hexdigest()
            if current_hash == expected_sha256:
                return destination
        temporary = destination.with_name(destination.name + ".part")
        request = urllib.request.Request(
            url, headers={"User-Agent": "upscayl-colab-notebook/1.0"}
        )
        try:
            with urllib.request.urlopen(request, timeout=180) as response, temporary.open("wb") as output:
                shutil.copyfileobj(response, output)
            if expected_sha256:
                actual_hash = hashlib.sha256(temporary.read_bytes()).hexdigest()
                if actual_hash.lower() != expected_sha256.lower():
                    raise RuntimeError(
                        f"SHA-256 ไม่ตรงสำหรับ {destination.name}: {actual_hash}"
                    )
            temporary.replace(destination)
        finally:
            if temporary.exists():
                temporary.unlink()
        return destination


    # Pin release เพื่อให้ได้ไบนารีที่ตรวจสอบ SHA-256 ได้
    UPSCAYL_CLI_TAG = "20251207-174704"
    CLI_URL = (
        "https://github.com/upscayl/upscayl-ncnn/releases/download/"
        f"{UPSCAYL_CLI_TAG}/upscayl-bin-{UPSCAYL_CLI_TAG}-linux.zip"
    )
    CLI_SHA256 = "a9fab3c770b62f2b7a35d8d6d61eb4e8b3aef79128b665c919d080b85a2292f2"
    CLI_ARCHIVE = RUNTIME_DIR / f"upscayl-bin-{UPSCAYL_CLI_TAG}-linux.zip"
    download_file(CLI_URL, CLI_ARCHIVE, CLI_SHA256)

    extract_dir = RUNTIME_DIR / "upscayl-bin-extracted"
    extract_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(CLI_ARCHIVE) as archive:
        root = extract_dir.resolve()
        for member in archive.infolist():
            target = (extract_dir / member.filename).resolve()
            if not target.is_relative_to(root):
                raise RuntimeError(f"พบ path ที่ไม่ปลอดภัยใน ZIP: {member.filename}")
        archive.extractall(extract_dir)

    executables = [path for path in extract_dir.rglob("upscayl-bin") if path.is_file()]
    if not executables:
        raise RuntimeError("ไม่พบไฟล์ upscayl-bin ใน official Linux release archive")
    shutil.copy2(executables[0], BIN_PATH)
    BIN_PATH.chmod(BIN_PATH.stat().st_mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)

    link_check = run_command(["ldd", str(BIN_PATH)], check=False)
    missing_libraries = [line.strip() for line in link_check.stdout.splitlines() if "not found" in line]
    if missing_libraries:
        raise RuntimeError("ไลบรารีที่ CLI ต้องใช้ยังไม่ครบ:\n" + "\n".join(missing_libraries))

    help_check = run_command([str(BIN_PATH), "-h"], check=False, env=os.environ.copy())
    if "Usage:" not in ((help_check.stdout or "") + (help_check.stderr or "")):
        print("⚠️ ดาวน์โหลด CLI แล้ว แต่ตรวจข้อความ help ไม่ได้; จะลองทำงานต่อ")

    print(f"✅ Upscayl NCNN CLI พร้อมใช้งาน: {BIN_PATH}")
    BACKEND_READY = True
    if callable(globals().get("set_status")):
        set_status(f"Upscayl พร้อมใช้งานบน {gpu_name}", "success")
    if globals().get("header") is not None:
        import html as _html
        header.value = header.value.replace("ตรวจสอบสถานะ backend ด้านล่าง", _html.escape(str(gpu_name)))
except Exception as setup_error:
    BACKEND_ERROR = f"{type(setup_error).__name__}: {setup_error}"
    print(f"⚠️ Upscayl backend ยังไม่พร้อม: {BACKEND_ERROR}")
    print("หน้า UI จะยังแสดง แต่ต้องแก้การตั้งค่า GPU/Vulkan ก่อนเริ่มประมวลผล")
    if callable(globals().get("set_status")):
        set_status(f"UI พร้อมแล้ว แต่ backend ยังไม่พร้อม: {BACKEND_ERROR}", "error")


### แหล่งที่มาและลิขสิทธิ์

Notebook ดาวน์โหลด Upscayl NCNN backend และคู่ไฟล์โมเดลจาก upstream ขณะใช้งาน โดยตรึงเวอร์ชัน CLI และ model commit ไว้ ดูข้อมูล upstream และเงื่อนไขการใช้งานที่ [README](README.md)
